# 01. Behavioral Validation of NPI Licensing

- **Project:** Mechanistic analysis of negative-polarity-item (NPI) licensing in Gemma-2-2B.
- **Stage:** Behavioral i.e. does the model distinguish licensed from unlicensed "any"?
- **Input:** npi_stimuli.csv (80 items, 5 licensing environments)
- **Outputs:** 01_behavioral_results.csv, 01_behavioral.png
- **Runtime:** ~20 min
- **Summary:** Establishes that Gemma-2-2B assigns higher probability to "any" when it is
structurally licensed (in-scope negation, "no", "few", questions, "without") than when
it is not.

In [ ]:
!pip install transformers accelerate torch --quiet
!pip install pandas matplotlib seaborn scipy --quiet

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from transformers import AutoTokenizer, AutoModelForCausalLM
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import login
login(os.environ["HF_TOKEN"])
print("Authenticated.")

In [ ]:
MODEL_NAME = "google/gemma-2-2b"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    dtype=torch.float16,
)
model.eval()
print("Model loaded successfully.")

In [ ]:
import glob, pandas as pd
csv_path = glob.glob("/kaggle/input/**/npi_stimuli.csv", recursive=True)[0]
stim_df = pd.read_csv(csv_path)
stimuli = list(stim_df[["item","condition","sentence","npi"]].itertuples(index=False, name=None))
item_info = stim_df.drop_duplicates("item").set_index("item")[["environment","design"]].to_dict("index")
triplet_items = stim_df[stim_df.design == "triplet"].item.unique()
print(f"Loaded from {csv_path}")
print(f"{len(stimuli)} sentences, {stim_df.item.nunique()} items across {stim_df.environment.nunique()} environments")

In [ ]:
def get_npi_log_prob(sentence, npi_word, model, tokenizer):
    # Tokenize the full sentence
    inputs = tokenizer(sentence, return_tensors="pt").to(model.device)
    input_ids = inputs["input_ids"][0]

    # Decode each token individually and find the NPI word by string match
    npi_position = None
    for i in range(len(input_ids)):
        decoded = tokenizer.decode([input_ids[i].item()]).strip().lower()
        if decoded == npi_word.lower():
            npi_position = i

    if npi_position is None:
        print(f"  WARNING: Could not find '{npi_word}' in: {sentence}")
        print(f"  Decoded tokens: {[tokenizer.decode([t.item()]).strip() for t in input_ids]}")
        return None, None

    # Forward pass
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0]

    # logits[i-1] predicts token[i]
    pred_position = npi_position - 1
    log_probs = torch.log_softmax(logits[pred_position], dim=-1)

    npi_token_id = input_ids[npi_position].item()
    npi_log_prob = log_probs[npi_token_id].item()

    return npi_log_prob, npi_position

In [ ]:
print("Running behavioral check...")

results = []
for item, condition, sentence, npi_word in stimuli:
    log_prob, position = get_npi_log_prob(sentence, npi_word, model, tokenizer)

    results.append({
        "item": item,
        "condition": condition,
        "sentence": sentence,
        "npi_word": npi_word,
        "log_prob": log_prob,
        "npi_position": position,
    })

    if len(results) % 15 == 0:
        print(f"  Processed {len(results)}/{len(stimuli)} sentences...")

df = pd.DataFrame(results)
print(f"\nDone. Processed {len(df)} sentences.")
print(f"Any failures: {df['log_prob'].isna().sum()}")

In [ ]:
print("Results: Mean log-probability of NPI by condition")

df["environment"] = df["item"].map(lambda x: item_info[x]["environment"])

# Overall means
summary = df.groupby("condition")["log_prob"].agg(["mean", "std", "count"])
print("\nOverall (all items):")
print(summary.round(4))

print("\nBy environment:")
print(df.groupby(["environment","condition"])["log_prob"].agg("mean").unstack().round(3))

In [ ]:
print("paired t-tests")

df_tri = df[df.item.isin(triplet_items)]

# Aligned by item, complete triplets only
tri = df_tri.pivot_table(index="item", columns="condition", values="log_prob").dropna(subset=["A","B","C"])
t_ab, p_ab = stats.ttest_rel(tri["A"], tri["B"])
t_bc, p_bc = stats.ttest_rel(tri["B"], tri["C"])

# A vs C uses ALL items, aligned by item
allac = df.pivot_table(index="item", columns="condition", values="log_prob").dropna(subset=["A","C"])
t_ac, p_ac = stats.ttest_rel(allac["A"], allac["C"])

def cohens_d_paired(x, y):
    diff = x - y
    return diff.mean() / diff.std()

print(f"\nA vs B (scope effect):      t={t_ab:.3f}, p={p_ab:.6f}")
print(f"  → Does negation SCOPE matter? (A should be > B)")
print(f"  → Mean diff: {(tri['A'] - tri['B']).mean():.4f}")

print(f"\nA vs C (licensing effect):  t={t_ac:.3f}, p={p_ac:.6f}")
print(f"  → Does licensing PRESENCE matter? (A should be > C)")
print(f"  → Mean diff: {(allac['A'] - allac['C']).mean():.4f}")

print(f"\nB vs C (residual negation): t={t_bc:.3f}, p={p_bc:.6f}")
print(f"  → Does out-of-scope negation differ from no licensor?")
print(f"  → Mean diff: {(tri['B'] - tri['C']).mean():.4f}")

print(f"\nEffect sizes (Cohen's d):")
print(f"  A vs B: {cohens_d_paired(tri['A'], tri['B']):.3f}")
print(f"  A vs C: {cohens_d_paired(allac['A'], allac['C']):.3f}")
print(f"  B vs C: {cohens_d_paired(tri['B'], tri['C']):.3f}")

print("\nPer-environment mean log P('any') by condition:")
print(df.groupby(["environment","condition"])["log_prob"].mean().unstack())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Overall distribution by condition
sns.boxplot(data=df, x="condition", y="log_prob", ax=axes[0],
            palette={"A": "#2ecc71", "B": "#e74c3c", "C": "#95a5a6"},
            order=["A", "B", "C"])
axes[0].set_title("Log P(NPI) by Condition\n(All items)", fontsize=12)
axes[0].set_xlabel("Condition")
axes[0].set_ylabel("Log probability of NPI word")
axes[0].axhline(y=0, color='black', linestyle='--', alpha=0.3)

labels = {"A": "In scope\n(grammatical)", "B": "Out of scope\n(ungrammatical)",
          "C": "Unlicensed\n(ungrammatical)"}
axes[0].set_xticklabels([f"{c}\n{labels[c]}" for c in ["A", "B", "C"]])

# Plot 2: items
sns.stripplot(data=df, x="condition", y="log_prob", ax=axes[1],
              palette={"A": "#2ecc71", "B": "#e74c3c", "C": "#95a5a6"},
              order=["A", "B", "C"], alpha=0.6, jitter=0.15)
sns.pointplot(data=df, x="condition", y="log_prob", ax=axes[1],
              color="black", order=["A", "B", "C"], markers="D",
              linestyles="--", errorbar=('ci',95), capsize=0.1)
axes[1].set_title("all items\nIndividual + Mean ± 95% CI", fontsize=12)
axes[1].set_xlabel("Condition")
axes[1].set_ylabel("Log probability")

# Plot 3: Item-by-item paired differences
item_diffs = df.pivot_table(index="item", columns="condition", values="log_prob")
tri_diffs = item_diffs.loc[item_diffs.index.isin(triplet_items)].copy()
tri_diffs["A_minus_B"] = tri_diffs["A"] - tri_diffs["B"]
tri_diffs["A_minus_C"] = tri_diffs["A"] - tri_diffs["C"]

axes[2].bar(range(len(tri_diffs)), tri_diffs["A_minus_B"],
            alpha=0.7, label="A − B (scope effect)", color="#3498db")
axes[2].bar(range(len(tri_diffs)), tri_diffs["A_minus_C"],
            alpha=0.5, label="A − C (negation effect)", color="#e67e22")
axes[2].axhline(y=0, color='black', linestyle='-', alpha=0.5)
axes[2].set_xlabel("Triplet item number")
axes[2].set_ylabel("Log-prob difference")
axes[2].set_title("Per-item effect sizes (triplet items)\n(positive = A wins, as expected)", fontsize=12)
axes[2].legend(fontsize=9)
axes[2].set_xticks(range(0, len(tri_diffs), 5))

plt.tight_layout()
plt.savefig("01_behavioral.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nPlot saved as 'phase1_results.png'")

In [ ]:
df.to_csv("01_behavioral_results.csv", index=False)
print("Results saved to 'phase1_results.csv'")
print("\nIf A >> B & A >> C, proceed to Phase 2.")